# Random Forest Models

This notebook version separates `target_refusal`, `target_capability` and `target_needs_clarification` cleanly, stores models by target and feature version, and keeps SHAP analysis target-specific.

In [ ]:
from pathlib import Path

import numpy as np
import pandas as pd
import shap
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (
    average_precision_score,
    f1_score,
    precision_score,
    recall_score,
    roc_auc_score,
)
from sklearn.metrics.pairwise import cosine_distances
from sklearn.model_selection import StratifiedKFold, cross_validate, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder

pd.set_option('display.float_format', lambda x: f'{x:.10f}')

In [7]:
RANDOM_STATE = 42
TEST_SIZE = 0.2
TARGETS = ['target_refusal', 'target_capability', 'target_needs_clarification']
VERSIONS = ['v01', 'v02', 'v03', 'v04']

PROJECT_ROOT = Path.cwd().parent
CONFIG = {
    'features_dir': PROJECT_ROOT / '01_data' / '03_features',
    'input_file': 'llm_sustainability_strict_v1-2-0_dataframe.csv',
    'embeddings_file': 'llm_sustainability_strict_v1-1-0_embeddings.npy'
}

INPUT_PATH = CONFIG['features_dir'] / CONFIG['input_file']
EMBEDDINGS_PATH = CONFIG['features_dir'] / CONFIG['embeddings_file']

In [8]:
def load_csv(path):
    with Path(path).open('r', encoding='utf-8') as file:
        return pd.read_csv(file)

def load_embeddings(path):
    with Path(path).open('rb') as file:
        return np.load(file)

def get_features_cols(features):
    features_cols = (
        features.get('num', [])
        + features.get('emb', [])
        + features.get('bin', [])
        + features.get('cat', [])
    )
    return list(dict.fromkeys(features_cols))

def create_preprocess(features):
    transformers = []
    if features.get('num'):
        transformers.append(('num', 'passthrough', features['num']))
    if features.get('emb'):
        transformers.append(('emb', 'passthrough', features['emb']))
    if features.get('bin'):
        transformers.append(('bin', 'passthrough', features['bin']))
    if features.get('cat'):
        transformers.append(('cat', OneHotEncoder(handle_unknown='ignore', sparse_output=False), features['cat']))
    return ColumnTransformer(transformers=transformers, remainder='drop')

def create_model():
    return RandomForestClassifier(
        n_estimators=500,
        max_features='sqrt',
        max_depth=None,
        min_samples_split=5,
        min_samples_leaf=2,
        class_weight='balanced',
        random_state=RANDOM_STATE,
        n_jobs=-1
    )

def original_feature_name(feature, features):
    if feature.startswith('num__'):
        return feature.removeprefix('num__')
    if feature.startswith('bin__'):
        return feature.removeprefix('bin__')
    if feature.startswith('emb__'):
        return 'embeddings'
    if feature.startswith('cat__'):
        feature_without_prefix = feature.removeprefix('cat__')
        categorical_columns = sorted(features.get('cat', []), key=len, reverse=True)
        for col in categorical_columns:
            if feature_without_prefix == col or feature_without_prefix.startswith(col + '_'):
                return col
        return feature_without_prefix
    return feature

In [9]:
df = load_csv(INPUT_PATH)
embeddings = load_embeddings(EMBEDDINGS_PATH)

df['prompt_style'] = df['prompt_style'].astype('category')
df['task_type'] = df['task_type'].astype('category')
df['topic_label'] = df['topic_label'].astype('category')
df['topic_cat'] = df['topic_cat'].astype('category')
df['target_refusal'] = df['target_refusal'].astype(int)
df['target_capability'] = df['target_capability'].astype(int)

embedding_cols = [f'embedding_{i}' for i in range(embeddings.shape[1])]

FEATURES_SELECTED_v01 = {
    'cat': [],
    'num': ['log_first_prompt_tokens', 'log_question_count', 'orthographic_error_rate'],
    'bin': ['has_role_instruction', 'has_audience_or_level_instruction', 'has_format_instruction'],
    'emb': []
}
FEATURES_SELECTED_v02 = {
    'cat': ['task_type', 'topic_cat'],
    'num': ['log_first_prompt_tokens', 'log_question_count', 'embedding_novelty', 'topic_prob', 'orthographic_error_rate'],
    'bin': ['has_role_instruction', 'has_audience_or_level_instruction', 'has_format_instruction'],
    'emb': []
}
FEATURES_SELECTED_v03 = {
    'cat': ['task_type', 'topic_cat'],
    'num': ['log_first_prompt_tokens', 'log_question_count', 'embedding_novelty', 'topic_prob', 'orthographic_error_rate'],
    'bin': ['has_role_instruction', 'has_audience_or_level_instruction', 'has_format_instruction'],
    'emb': embedding_cols
}
FEATURES_SELECTED_v04 = {
    'cat': [],
    'num': [],
    'bin': [],
    'emb': embedding_cols
}
FEATURES_SELECTIONS = {
    'v01': FEATURES_SELECTED_v01,
    'v02': FEATURES_SELECTED_v02,
    'v03': FEATURES_SELECTED_v03,
    'v04': FEATURES_SELECTED_v04
}
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print('Input path:', INPUT_PATH)
print('Data shape:', df.shape)
print('Embeddings shape:', embeddings.shape)

Input path: c:\Users\heike\Desktop\Stackfuel\Portfolio\llm-sustainability-analysis\01_data\03_features\llm_sustainability_strict_v1-2-0_dataframe.csv
Data shape: (46429, 37)
Embeddings shape: (46429, 384)


In [10]:
datasets = {}
fitted_pipelines = {}
features_by_version = {}
results_by_target = {}

for target in TARGETS:
    train_idx, test_idx = train_test_split(
        np.arange(len(df)),
        test_size=TEST_SIZE,
        random_state=RANDOM_STATE,
        stratify=df[target]
    )

    x_train_df_target = df.iloc[train_idx].copy()
    x_test_df_target = df.iloc[test_idx].copy()
    x_train_embeddings_target = embeddings[train_idx]
    x_test_embeddings_target = embeddings[test_idx]

    centroid = x_train_embeddings_target.mean(axis=0)
    x_train_df_target['embedding_novelty'] = cosine_distances(x_train_embeddings_target, centroid.reshape(1, -1)).flatten()
    x_test_df_target['embedding_novelty'] = cosine_distances(x_test_embeddings_target, centroid.reshape(1, -1)).flatten()

    x_train_embeddings_df = pd.DataFrame(x_train_embeddings_target, columns=embedding_cols, index=x_train_df_target.index)
    x_test_embeddings_df = pd.DataFrame(x_test_embeddings_target, columns=embedding_cols, index=x_test_df_target.index)

    x_train_full_target = pd.concat([x_train_df_target, x_train_embeddings_df], axis=1)
    x_test_full_target = pd.concat([x_test_df_target, x_test_embeddings_df], axis=1)

    datasets[target] = {
        'x_train_df': x_train_df_target,
        'x_test_df': x_test_df_target,
        'x_train_full': x_train_full_target,
        'x_test_full': x_test_full_target,
        'train_idx': train_idx,
        'test_idx': test_idx
    }

    fitted_pipelines[target] = {}
    features_by_version[target] = {}
    results_by_target[target] = {}

    y_train = x_train_df_target[target]
    y_test = x_test_df_target[target]

    for version, features in FEATURES_SELECTIONS.items():
        features_cols = get_features_cols(features)
        X_train_version = x_train_full_target[features_cols]
        X_test_version = x_test_full_target[features_cols]

        pipeline = Pipeline([('preprocess', create_preprocess(features)), ('model', create_model())])

        results = cross_validate(
            pipeline,
            X_train_version,
            y_train,
            cv=cv,
            scoring={
                'roc_auc': 'roc_auc',
                'average_precision': 'average_precision',
                'f1': 'f1',
                'precision': 'precision',
                'recall': 'recall'
            },
            n_jobs=1
        )

        pipeline.fit(X_train_version, y_train)
        test_pred = pipeline.predict(X_test_version)
        test_proba = pipeline.predict_proba(X_test_version)[:, 1]

        results_by_target[target][version] = {
            'cv_roc_auc': results['test_roc_auc'].mean(),
            'cv_average_precision': results['test_average_precision'].mean(),
            'cv_f1': results['test_f1'].mean(),
            'cv_precision': results['test_precision'].mean(),
            'cv_recall': results['test_recall'].mean(),
            'test_roc_auc': roc_auc_score(y_test, test_proba),
            'test_average_precision': average_precision_score(y_test, test_proba),
            'test_f1': f1_score(y_test, test_pred),
            'test_precision': precision_score(y_test, test_pred),
            'test_recall': recall_score(y_test, test_pred)
        }

        fitted_pipelines[target][version] = pipeline
        features_by_version[target][version] = features_cols.copy()

        print(target, version)
        print(results_by_target[target][version])
        print('##########')

target_refusal v01
{'cv_roc_auc': np.float64(0.6072625911540532), 'cv_average_precision': np.float64(0.07420148685199082), 'cv_f1': np.float64(0.08050861908569444), 'cv_precision': np.float64(0.04788721567873224), 'cv_recall': np.float64(0.25384944396920445), 'test_roc_auc': 0.6749658964511323, 'test_average_precision': 0.11528780755489937, 'test_f1': 0.11194539249146758, 'test_precision': 0.0653386454183267, 'test_recall': 0.3904761904761905}
##########
target_refusal v02
{'cv_roc_auc': np.float64(0.7010339429912162), 'cv_average_precision': np.float64(0.17965112537509248), 'cv_f1': np.float64(0.22110256742785914), 'cv_precision': np.float64(0.5991328226622344), 'cv_recall': np.float64(0.13585685771314512), 'test_roc_auc': 0.740466746416504, 'test_average_precision': 0.29050971184092206, 'test_f1': 0.3490909090909091, 'test_precision': 0.7384615384615385, 'test_recall': 0.22857142857142856}
##########
target_refusal v03
{'cv_roc_auc': np.float64(0.8039144350009986), 'cv_average_precis

In [11]:
results_rows = []
for target in TARGETS:
    for version in VERSIONS:
        row = {'target': target, 'version': version}
        row.update(results_by_target[target][version])
        results_rows.append(row)

results_df = pd.DataFrame(results_rows)
results_df

,target,version,cv_roc_auc,cv_average_precision,cv_f1,cv_precision,cv_recall,test_roc_auc,test_average_precision,test_f1,test_precision,test_recall
0,target_refusal,v01,0.6072625912,0.0742014869,0.0805086191,0.0478872157,0.2538494440,0.6749658965,0.1152878076,0.1119453925,0.0653386454,0.3904761905
1,target_refusal,v02,0.7010339430,0.1796511254,0.2211025674,0.5991328227,0.1358568577,0.7404667464,0.2905097118,0.3490909091,0.7384615385,0.2285714286
2,target_refusal,v03,0.8039144350,0.2607025553,0.3024855745,0.6305974527,0.1990162532,0.8273418645,0.3739010659,0.4275862069,0.7750000000,0.2952380952
3,target_refusal,v04,0.7970955424,0.2308528567,0.2935786265,0.5512057258,0.2002067294,0.8301548826,0.3240059407,0.4117647059,0.6562500000,0.3000000000
4,target_capability,v01,0.6902238603,0.0543728077,0.0631497722,0.0347901704,0.3423576424,0.7199323934,0.0543048253,0.0579964851,0.0317002882,0.3402061856
5,target_capability,v02,0.7296627613,0.1911228419,0.2563526361,0.7985859729,0.1529470529,0.7900835041,0.2626885135,0.2477876106,0.8750000000,0.1443298969
6,target_capability,v03,0.8239949878,0.2805102409,0.3261952518,0.7438405797,0.2099567100,0.8285388289,0.3536814159,0.4031007752,0.8125000000,0.2680412371
7,target_capability,v04,0.8064583653,0.2421621410,0.3057907678,0.5645754246,0.2099567100,0.8076022093,0.2818198513,0.3636363636,0.5652173913,0.2680412371
8,target_needs_clarification,v01,0.6686863860,0.3282540432,0.3426740160,0.3077720379,0.3868404248,0.6697198248,0.3364167514,0.3416067929,0.3047785548,0.3885586924
9,target_needs_clarification,v02,0.7850753892,0.4987602415,0.4295182881,0.6257958247,0.3272007124,0.7956693683,0.5152299358,0.4533205005,0.6434426230,0.3499257058


# Summary of Random Forest Results

Random Forest classifiers were trained for three highly imbalanced target variables: `target_refusal`, `target_capability`, and `target_needs_clarification`. For each target, four feature configurations were compared:

- **v01:** Structural and linguistic prompt features.
- **v02:** Structural features plus engineered semantic features.
- **v03:** Structural features, engineered semantic features, and dense embeddings.
- **v04:** Dense embeddings only.

Because the target variables have different class distributions, target-specific stratified train-test splits were used. Within each target, the same split was used for all four feature configurations to ensure a fair comparison.

## Model Performance

| Target | Best model | CV ROC-AUC | Test ROC-AUC | Test Average Precision | Test F1 | Test Precision | Test Recall |
|---|---:|---:|---:|---:|---:|---:|---:|
| `target_refusal` | v03 | 0.804 | 0.827 | 0.374 | 0.428 | 0.775 | 0.295 |
| `target_capability` | v03 | 0.824 | 0.829 | 0.354 | 0.403 | 0.813 | 0.268 |
| `target_needs_clarification` | v03 | 0.859 | 0.861 | 0.626 | 0.457 | 0.874 | 0.310 |

## Interpretation

The results show a consistent performance improvement when moving from structural and linguistic features to semantic representations.

The baseline model v01 provides only limited predictive performance for all three targets. This indicates that surface-level prompt characteristics alone are insufficient to reliably distinguish the positive cases.

Adding engineered semantic features in v02 leads to a substantial improvement in ranking performance and precision. However, v02 often operates conservatively, identifying only a relatively small proportion of the positive cases at the selected classification threshold.

The full feature model v03 achieves the best overall performance for all three targets. It obtains the highest or near-highest values for ROC-AUC, Average Precision, F1, and precision. This indicates that dense embeddings provide the dominant predictive signal, while the additional structural and engineered semantic features contribute complementary information.

The embeddings-only model v04 also performs well, particularly for ranking the observations. However, it generally performs worse than v03 in terms of precision, F1, and Average Precision. This suggests that the handcrafted and engineered semantic features improve the quality of positive predictions beyond the information captured by the embeddings alone.

## Target-Specific Findings

For `target_refusal`, v03 achieves the best balance between precision and recall. Although v04 has a marginally higher test ROC-AUC and recall, v03 performs better in terms of Average Precision, F1, and precision.

For `target_capability`, v03 clearly outperforms the other feature configurations. It achieves a test precision of 0.813 and an Average Precision of 0.354, while maintaining the same recall as v04. The additional features therefore make the positive predictions considerably more reliable.

For `target_needs_clarification`, all models perform better than for the other targets. v03 achieves a test ROC-AUC of 0.861, an Average Precision of 0.626, and a precision of 0.874. v04 performs similarly, but v03 remains the strongest overall model.

## Overall Conclusion

Overall, v03 is the preferred model configuration for all three target variables:

> Dense embeddings represent the main source of predictive information, while structural and engineered semantic features provide meaningful complementary value.

The models achieve high precision but relatively moderate recall for `target_refusal` and `target_capability`. Therefore, they are well suited for ranking and prioritizing cases for further analysis, but they do not yet identify all positive cases. If higher recall is required, the classification threshold should be optimized using a validation set and the corresponding precision-recall trade-off should be reported.

The SHAP analysis further supports this interpretation: the embedding block accounts for the largest share of the model's predictive contribution. However, this result is partly influenced by the high dimensionality of the embedding representation. The individual embedding dimensions are not directly interpretable, whereas the engineered features allow more transparent feature-level and feature-group-level explanations.

In [14]:
# SHAP for v03

def calculate_shap_importance(target, version='v03', n_sample=2000, n_background=100):
    pipeline = fitted_pipelines[target][version]
    features_cols = features_by_version[target][version]
    features = FEATURES_SELECTIONS[version]
    model = pipeline.named_steps['model']
    preprocess = pipeline.named_steps['preprocess']

    X_train_version = datasets[target]['x_train_full'][features_cols]
    X_test_version = datasets[target]['x_test_full'][features_cols]
    feature_names = preprocess.get_feature_names_out()

    n_sample = min(n_sample, len(X_test_version))
    n_background = min(n_background, len(X_train_version))

    X_shap_sample = X_test_version.sample(n=n_sample, random_state=42)
    X_background = X_train_version.sample(n=n_background, random_state=42)

    X_shap_transformed = preprocess.transform(X_shap_sample)
    X_background_transformed = preprocess.transform(X_background)

    explainer = shap.TreeExplainer(model, data=X_background_transformed, feature_perturbation='interventional')
    shap_exp = explainer(X_shap_transformed, check_additivity=False)
    shap_values = shap_exp.values

    if isinstance(shap_values, list):
        shap_values = np.asarray(shap_values[1])
    if shap_values.ndim == 3:
        shap_values = shap_values[:, :, 1]

    fi_shap = pd.DataFrame({
        'feature': feature_names,
        'importance': np.abs(shap_values).mean(axis=0)
    })
    fi_shap['original_feature'] = fi_shap['feature'].apply(lambda feature: original_feature_name(feature, features))

    grouped_importance = (
        fi_shap
        .groupby('original_feature')
        .agg(
            importance_sum=('importance', 'sum'),
            importance_mean=('importance', 'mean'),
            n_dimensions=('importance', 'count')
        )
        .sort_values('importance_sum', ascending=False)
    )
    grouped_importance['importance_share'] = grouped_importance['importance_sum'] / grouped_importance['importance_sum'].sum()
    return fi_shap, grouped_importance

# Example:
fi_shap_v03_capability, grouped_v03_capability = calculate_shap_importance('target_capability', 'v03')
print(grouped_v03_capability.to_string(float_format=lambda x: f'{x:.10f}'))

100%|===================| 3998/4000 [08:14<00:00]        

                                   importance_sum  importance_mean  n_dimensions  importance_share
original_feature                                                                                  
embeddings                           0.0518437574     0.0001350098           384      0.9574013545
log_first_prompt_tokens              0.0014177987     0.0014177987             1      0.0261825625
task_type                            0.0003680889     0.0000408988             9      0.0067975176
embedding_novelty                    0.0002253671     0.0002253671             1      0.0041618667
orthographic_error_rate              0.0000981915     0.0000981915             1      0.0018133083
topic_prob                           0.0000615093     0.0000615093             1      0.0011358950
log_question_count                   0.0000514813     0.0000514813             1      0.0009507077
topic_cat                            0.0000503481     0.0000125870             4      0.0009297804
has_format

In [17]:
fi_shap_v03_refusal, grouped_v03_refusal = calculate_shap_importance('target_refusal', 'v03')
print(grouped_v03_refusal.to_string(float_format=lambda x: f'{x:.10f}'))

100%|===================| 3993/4000 [17:07<00:01]        

                                   importance_sum  importance_mean  n_dimensions  importance_share
original_feature                                                                                  
embeddings                           0.0810328833     0.0002110231           384      0.9747676398
embedding_novelty                    0.0006760997     0.0006760997             1      0.0081329961
log_first_prompt_tokens              0.0006643244     0.0006643244             1      0.0079913472
task_type                            0.0003063900     0.0000340433             9      0.0036856524
orthographic_error_rate              0.0001241683     0.0001241683             1      0.0014936559
topic_cat                            0.0001181949     0.0000295487             4      0.0014217997
topic_prob                           0.0001044304     0.0001044304             1      0.0012562236
log_question_count                   0.0000671382     0.0000671382             1      0.0008076250
has_format

In [2]:
# fi_shap_v03_needs_clarification, grouped_v03_needs_clarification = calculate_shap_importance('target_needs_clarification', 'v03')
# print(grouped_v03_needs_clarification.to_string(float_format=lambda x: f'{x:.10f}'))